<a href="https://colab.research.google.com/github/springboardmentor787-stack/Company-Internal-Chatbot-with-Role-Based-Access-Control-RBAC---Group-1/blob/Srija-Mitra/Internal_Chatbot_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install fastapi streamlit langchain-community sentence-transformers chromadb pandas langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 

In [2]:
!git clone https://github.com/springboardmentor441p-coderr/Fintech-data.git Fintech-data-main

Cloning into 'Fintech-data-main'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 43 (delta 2), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 51.89 KiB | 3.46 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [3]:
!ls Fintech-data-main


engineering  Finance  general  HR  marketing


In [18]:
role_access_mapping = {
    "Finance": ["Finance", "C-Level"],
    "marketing": ["Marketing", "C-Level"],
    "HR": ["HR", "C-Level"],
    "engineering": ["Engineering", "C-Level"],
    "general": ["Finance", "Marketing", "HR", "Engineering", "C-Level", "General"]
}


In [5]:
from langchain_community.document_loaders import TextLoader, CSVLoader
from pathlib import Path

file_path = Path("Fintech-data-main") / "engineering" / "engineering_master_doc.md"
loader = TextLoader(str(file_path), encoding="utf-8", autodetect_encoding=True)


docs = loader.load()
print(docs)

[Document(metadata={'source': 'Fintech-data-main/engineering/engineering_master_doc.md'}, page_content='# FinSolve Technologies Engineering Document\n\n## 1. Introduction\n\n### 1.1 Company Overview\nFinSolve Technologies is a leading FinTech company headquartered in Bangalore, India, with operations across North America, Europe, and Asia-Pacific. Founded in 2018, FinSolve provides innovative financial solutions, including digital banking, payment processing, wealth management, and enterprise financial analytics, serving over 2 million individual users and 10,000 businesses globally.\n\n### 1.2 Purpose\nThis engineering document outlines the technical architecture, development processes, and operational guidelines for FinSolve\'s product ecosystem. It serves as a comprehensive guide for engineering teams, stakeholders, and partners to ensure alignment with FinSolve\'s mission: "To empower financial freedom through secure, scalable, and innovative technology solutions."\n\n### 1.3 Scope

In [20]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, CSVLoader

DATA_ROOT = Path("Fintech-data-main")
all_documents = []

for department, roles in role_access_mapping.items():
    dept_path = DATA_ROOT / department

    for file in dept_path.glob("*"):


        if file.suffix == ".md":
            loader = TextLoader(str(file), encoding="utf-8")
        elif file.suffix == ".csv":
            loader = CSVLoader(str(file))
        else:
            continue

        docs = loader.load()

        for d in docs:
            d.page_content = clean_text(d.page_content)


        for d in docs:
            d.metadata = {
                "department": department,
                "allowed_roles": roles,
                "source": file.name
            }

        all_documents.extend(docs)

print("Total loaded documents:", len(all_documents))
print("Sample metadata:", all_documents[0].metadata)

Total loaded documents: 109
Sample metadata: {'department': 'Finance', 'allowed_roles': ['Finance', 'C-Level'], 'source': 'financial_summary.md'}


In [19]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s.,]", "", text)
    return text.strip()


In [21]:
import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_documents(all_documents)

print("Total chunks:", len(chunks))
print("Sample chunk metadata:", chunks[0].metadata)

Total chunks: 315
Sample chunk metadata: {'department': 'Finance', 'allowed_roles': ['Finance', 'C-Level'], 'source': 'financial_summary.md'}


In [22]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


for chunk in chunks:
    if isinstance(chunk.metadata.get('allowed_roles'), list):
        chunk.metadata['allowed_roles'] = ",".join(chunk.metadata['allowed_roles'])


vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

In [23]:
!ls chroma_db

c5b146c8-24ab-4a39-8a81-fa9313e6045d  chroma.sqlite3


In [24]:

query = "What is our quarterly revenue and financial performance?"

hr_filter = {
    "allowed_roles": { "$eq": "HR" }
}

results = vector_db.similarity_search(query, k=3, filter=hr_filter)

print(f"Query: {query}")
print(f"User Role: HR")
print(f"Results found: {len(results)}")

if len(results) == 0:
    print("Access Denied. HR users cannot see Finance data.")
else:
    print("Sensitive data was leaked!")

Query: What is our quarterly revenue and financial performance?
User Role: HR
Results found: 0
Access Denied. HR users cannot see Finance data.


In [25]:

finance_filter = {
    "allowed_roles": { "$eq": "Finance,C-Level" }
}

authorized_results = vector_db.similarity_search(query, k=3, filter=finance_filter)

print(f"\nQuery: {query}")
print(f"User Role: Finance")
print(f"Results found: {len(authorized_results)}")

if len(authorized_results) > 0:
    print("Data retrieved for authorized user.")


Query: What is our quarterly revenue and financial performance?
User Role: Finance
Results found: 3
Data retrieved for authorized user.


In [26]:
test_roles = ["HR", "Engineering", "Marketing", "Finance", "C-Level", "General"]
finance_query = "What is our quarterly revenue and financial performance?"

print(f"--- SECURITY AUDIT: {finance_query} ---")

role_to_allowed_strings = {}
all_unique_allowed_role_metadata_strings = set()
for roles_list_from_mapping in role_access_mapping.values():
    all_unique_allowed_role_metadata_strings.add(",".join(roles_list_from_mapping))

for single_role_to_test in test_roles:
    matching_allowed_strings = []
    for allowed_role_metadata_str in all_unique_allowed_role_metadata_strings:
        if single_role_to_test in allowed_role_metadata_str.split(','):
            if single_role_to_test not in ["Finance", "C-Level"] and "Finance" in allowed_role_metadata_str.split(','):
                continue
            matching_allowed_strings.append(allowed_role_metadata_str)
    role_to_allowed_strings[single_role_to_test] = matching_allowed_strings

for role in test_roles:
    allowed_strings_for_this_role = role_to_allowed_strings.get(role, [])

    if role not in ["Finance", "C-Level"]:
        allowed_strings_for_this_role = ["__NO_MATCH_POSSIBLE__"]

    if not allowed_strings_for_this_role:
        role_filter = {"allowed_roles": {"$in": ["__NO_MATCH_PLACEHOLDER__"]}}
    else:
        role_filter = {"allowed_roles": {"$in": allowed_strings_for_this_role}}

    test_results = vector_db.similarity_search(finance_query, k=3, filter=role_filter)

    expected_to_find_results = (role == "Finance" or role == "C-Level")

    if expected_to_find_results:
        status = "ACCESS GRANTED" if len(test_results) > 0 else "FAILURE: No data found for authorized role"
    else:
        status = "ACCESS DENIED" if len(test_results) == 0 else "SECURITY LEAK DETECTED"

    print(f"Role: {role:<12} | Results: {len(test_results)} | Status: {status}")

--- SECURITY AUDIT: What is our quarterly revenue and financial performance? ---
Role: HR           | Results: 0 | Status: ACCESS DENIED
Role: Engineering  | Results: 0 | Status: ACCESS DENIED
Role: Marketing    | Results: 0 | Status: ACCESS DENIED
Role: Finance      | Results: 3 | Status: ACCESS GRANTED
Role: C-Level      | Results: 3 | Status: ACCESS GRANTED
Role: General      | Results: 0 | Status: ACCESS DENIED
